<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_6_NeuralImportanceSampling_Asimov_SameSampleNorm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 6 — Neural importance sampling with same-sample normalization

Retrain the proposal with globally normalized likelihood weights, then
fine-tune its conditional variance objective. Compare a wider defensive
mixture grid for both $q_{0,A}$ and the full $t_A(\mu)$ scan.

The Exercise 5 models and pilot target are reused. New proposal checkpoints,
seeds, plots, and results use the v2 directories. The two previous proposals
are evaluated on the same tuning sample when their saved checkpoints exist.
The original scan figures and timing benchmark are retained.


In [ ]:
## ============================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False
# ----------------------------------------------------------------------------

import subprocess
from pathlib import Path

DEPENDENCIES = [
    "pytorch-lightning",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "iminuit",
    "mplhep",
    "nflows",
    "pyarrow",
]


def run(*args, env=None):
    """Run one setup command and stop immediately if it fails."""
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    # Keep the checkout where earlier notebook versions placed it. The source
    # helpers now live in ml4hep_tifr_colab, while the untracked legacy
    # ml4hep_tifr directory remains the persistent workspace for data/models.
    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
        run(
            "git", "-C", REPO_DIR, "sparse-checkout", "set",
            "src", "workshops/ml4hep_tifr_colab",
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        #run("git", "-C", REPO_DIR, "checkout", BRANCH)
        #run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)

    #run(
    #    "git", "-C", REPO_DIR, "sparse-checkout", "set",
    #    "src", "workshops/ml4hep_tifr_colab",
    #)

    # Import package code and tutorial-local helpers from their new locations.
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)

    run(sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES)

    # Preserve the exact old working location. Existing dataframes, models,
    # densities, and plots are reused; only a missing directory is created.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)

    # Refresh these helpers from the fetched branch without touching model files
    # or local edits in the tutorial checkout. Re-run this cell after git updates.
    HELPER_DIR = WORK_DIR / "exercise6_same_sample_norm_v2_helpers"
    HELPER_DIR.mkdir(exist_ok=True)
    for module_name in ("utils_nf", "utils_nis"):
        helper_source = subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "show",
             f"FETCH_HEAD:workshops/ml4hep_tifr_colab/{module_name}.py"],
            text=True,
        )
        (HELPER_DIR / f"{module_name}.py").write_text(helper_source)
        sys.modules.pop(module_name, None)
    helper_path = str(HELPER_DIR.resolve())
    if helper_path in sys.path:
        sys.path.remove(helper_path)
    sys.path.insert(0, helper_path)

    remake_events = globals().get("REMAKE_EVENTS", False)
    cached_preselection = any(
        (Path(directory) / "exercise5_preselection_state.npz").exists()
        for directory in (
            "saved_asimov_nis_same_sample_norm_v2",
            "saved_asimov_nis_same_sample_norm",
            "saved_asimov_nis_influence_v2",
        )
    )
    missing_events = any(
        not Path(f"dataframes/{sample}.parquet").exists()
        for sample in ("signal", "background")
    )
    if remake_events or (missing_events and not cached_preselection):
        run(
            sys.executable,
            TUTORIAL_DIR / "generate_distributions.py",
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
        )

print("Working dir:", os.getcwd())



## Statistical target

Write $R_s=r_{s,\boldsymbol{\psi}}/\mathbb E_{q_\phi}[r_{s,\boldsymbol{\psi}}]$
for the population-normalized process ratios. This yield-only model has

$$
h_\mu(\mathbf{x})=\mu\lambda_S R_S(\mathbf{x})+\lambda_B R_B(\mathbf{x}),
\qquad h_A=h_{\mu_A},\qquad \mu_A=1.
$$

Define $L_\mu=\log(h_A/h_\mu)$, $Y_\mu=h_A L_\mu$, and
$I_\mu=\mathbb E_{q_\phi}[Y_\mu]$. The expected statistic is
$t_A(\mu)=2[(\mu-\mu_A)\lambda_S+I_\mu]$.

The signal and background normalizers are shared across the yield scan.
Combining their generating-point and tested-point contributions gives

$$
\begin{aligned}
b_S(\mu)&=\lambda_S\mathbb E_{q_\phi}\!\left[
R_S\left\{\mu_A(L_\mu+1)-\mu\frac{h_A}{h_\mu}\right\}\right],\\
b_B(\mu)&=\lambda_B\mathbb E_{q_\phi}\!\left[
R_B\left\{L_\mu+1-\frac{h_A}{h_\mu}\right\}\right],\\
\Psi_\mu(\mathbf{x})&=Y_\mu(\mathbf{x})-I_\mu
-b_S(\mu)[R_S(\mathbf{x})-1]-b_B(\mu)[R_B(\mathbf{x})-1].
\end{aligned}
$$

Thus $\mathbb E_{q_\phi}[\Psi_\mu]=0$. For independent quadrature points from $g$,
with both process ratios normalized on that same weighted sample,

$$
\operatorname{Var}_g[\widehat t_{A,M}(\mu)]
=\frac{4}{M}\int\frac{q_\phi^2(\mathbf{x})}{g(\mathbf{x})}
\Psi_\mu^2(\mathbf{x})\,d\mathbf{x}+o(M^{-1}).
$$

For the same equally weighted scan grid as Exercise 6, use

$$
A_{\mathrm{norm}}(\mathbf{x})=
\left[\frac1K\sum_{k=1}^K
\left(\frac{\Psi_{\mu_k}(\mathbf{x})}{\lambda_A}\right)^2\right]^{1/2},
\qquad
g^*(\mathbf{x})\propto q_\phi(\mathbf{x})A_{\mathrm{norm}}(\mathbf{x}),
$$

where $\lambda_A=\mu_A\lambda_S+\lambda_B$ is an irrelevant common scale.
The pilot fixes all expectations entering this target. The proposal flow
$g_{\boldsymbol{\eta}}$ is trained with weights proportional to its regularized
amplitude. We retain the defensive mixture
$g_\epsilon=(1-\epsilon)g_{\boldsymbol{\eta}}+\epsilon q_\phi$.


In [ ]:
import gc
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from scipy.special import logsumexp

import torch

from utils_nis import (
    epsilon_diagnostics, finetune_variance, mixture_log_weights, mix_quadratures,
)

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_plotting import export_standalone_figure_script
from utils_nf import (
    accumulate_preselection_histogram,
    checkpoint_path,
    choose_preselection_ratio_cut,
    flow_log_prob_x,
    flow_sample_x,
    load_flow,
    train_flow,
)

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)
SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Features: {FEATURES}")


## Configuration and saved checkpoints

Keep the Exercise 5 inputs and original scan grids. The v2 directories trigger
fresh proposal training on the first run and allow checkpoint reuse later.
$128$ repetitions use a new study seed. Set RUN_VARIANCE_FINETUNING=False
to evaluate the corrected likelihood fit alone.


In [ ]:
BASE_PATH = Path("./dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}

ORIGINAL_NIS_MODEL_DIR = Path("models_flows_asimov_nis_influence_v2")
ORIGINAL_NIS_CACHE_DIR = Path("saved_asimov_nis_influence_v2")
PREVIOUS_NIS_MODEL_DIR = Path("models_flows_asimov_nis_same_sample_norm")
NIS_TRAIN_SEED = SEED + 1201
STUDY_SEED = SEED + 100_000
TUNING_SEED = SEED + 2400
RUN_VARIANCE_FINETUNING = True
EPSILON_OBJECTIVE = "scan"  # "scan" or "q0"

NIS_MODEL_DIR = Path("models_flows_asimov_nis_same_sample_norm_v2_mle")
NIS_VARIANCE_MODEL_DIR = Path("models_flows_asimov_nis_same_sample_norm_v2_variance")
NIS_PLOT_DIR = Path("plots_asimov_nis_same_sample_norm_v2")
NIS_CACHE_DIR = Path("saved_asimov_nis_same_sample_norm_v2")
FIGURE_SCRIPT_DIR = Path("exercise6_same_sample_norm_v2_figures_scripts")
for directory in [NIS_MODEL_DIR, NIS_PLOT_DIR, NIS_CACHE_DIR, FIGURE_SCRIPT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}

# Exercise 5 PRESEL definition.
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)

# Saved Exercise 5 ensemble and flow architecture.
RATIO_ENSEMBLE_SIZE = 4
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65_536

# Neural importance-sampling design.
ASIMOV_MU_TRUE = 1.0
MU_DESIGN = np.linspace(0.0, 3.0, 13)
MU_SCAN = np.linspace(0.0, 3.0, 61)
PILOT_EVENTS = 2_000_000
NIS_TARGET_CLIP_QUANTILE = 0.9999
NIS_TARGET_FLOOR_FRACTION = 1.0e-5
DEFENSIVE_REFERENCE_FRACTION = 0.10  # selected from the tuning grid below
DEFENSIVE_FRACTION_CANDIDATES = (0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0)
TUNING_EVENTS = 200_000
VARIANCE_TRAINING_CONFIG = {
    "epochs": 20, "batch_size": 4096, "learning_rate": 1.0e-5, "patience": 5,
}
ACCEPTANCE_CALIBRATION_EVENTS = 250_000

# Proof-of-principle study.
BENCHMARK_EVENTS = 1_000_000
BENCHMARK_BLOCKS = 10
STUDY_SAMPLE_SIZES = np.asarray(
    [512, 1_024, 2_048, 4_096, 8_192, 16_384, 32_768], dtype=int
)
N_REPETITIONS = 128
SHOWCASE_SAMPLE_SIZE = 2_048

# This deliberately aggressive proposal has capacity comparable to the
# reference flow. Exact importance weights still correct residual errors.
NIS_MODEL_CONFIG = {
    "flow_type": "quadratic_spline",
    "n_features": N_DIM,
    "n_coupling_layers": 12,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "scale_clip": 1.5,
    "spline_num_bins": 24,
    "spline_tail_bound": 6.0,
    "dropout_probability": 0.0,
}
NIS_TRAINING_CONFIG = {
    "batch_size": 4096,
    "n_epochs": 70,
    "learning_rate": 1.0e-4,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-7,
    "weight_decay": 0.0,
    "validation_fraction": 0.20,
    "patience": 10,
    "gradient_clip": 5.0,
}


def export_exercise6_figure(fig, script_name):
    """Write a self-contained, editable script for one completed figure."""
    return export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )


print(f"Standalone figure scripts will be written to {FIGURE_SCRIPT_DIR}/")

required_paths = [
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
    checkpoint_path("reference", REFERENCE_FLOW_MODEL_DIR, REFERENCE_FLOW_TYPE),
]
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    for member in range(RATIO_ENSEMBLE_SIZE):
        required_paths.extend(
            [model_dir / f"model{member}.onnx", model_dir / f"model_scaler{member}.bin"]
        )

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    formatted = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Exercise 6 only loads Exercise 5 models. Missing checkpoints:\n"
        f"{formatted}\nRun Exercise 5 through both ratio trainings first."
    )
print("All Exercise 5 checkpoints are available.")


## Load the PRESEL classifier and saved selection

Reuse the original Exercise 6 PRESEL cut and yields when available. Otherwise
reconstruct them with the same streamed histogram pass as Exercise 6.


In [ ]:
def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


PRESEL_scaler, PRESEL_model_proto = load_trained_model(
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
)
PRESEL_model = as_inference_session(PRESEL_model_proto)
del PRESEL_model_proto


def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False),
        scaler=PRESEL_scaler,
        model=PRESEL_model,
    )
    return np.asarray(ratio, dtype=np.float64).reshape(-1)


PRESEL_STATE_PATH = NIS_CACHE_DIR / "exercise5_preselection_state.npz"
preselection_source = PRESEL_STATE_PATH
for cache_dir in (
    NIS_CACHE_DIR, Path("saved_asimov_nis_same_sample_norm"), ORIGINAL_NIS_CACHE_DIR
):
    candidate = cache_dir / PRESEL_STATE_PATH.name
    if candidate.exists():
        preselection_source = candidate
        break
if preselection_source.exists():
    state = np.load(preselection_source)
    PRESEL_RATIO_CUT = float(state["ratio_cut"])
    LAM_SIG = float(state["lambda_signal"])
    LAM_BKG = float(state["lambda_background"])
    print(f"Loaded cached PRESEL state from {preselection_source}")
else:
    edges = np.linspace(
        PRESEL_LOG_RATIO_RANGE[0],
        PRESEL_LOG_RATIO_RANGE[1],
        PRESEL_CUT_HISTOGRAM_BINS + 1,
    )
    histograms = {}
    statistics = {}
    for sample_name in ["signal", "background"]:
        histograms[sample_name], statistics[sample_name] = (
            accumulate_preselection_histogram(
                SAMPLE_PATHS[sample_name],
                features=FEATURES,
                ratio_predictor=evaluate_PRESEL_ratio,
                log_ratio_edges=edges,
                batch_size=STREAM_BATCH_SIZE,
                presel_fraction=PRESEL_TRAIN_FRACTION,
                flow_train_fraction=FLOW_TRAIN_FRACTION,
                split_seed=SPLIT_SEED,
            )
        )

    PRESEL_RATIO_CUT, diagnostics = choose_preselection_ratio_cut(
        histograms["signal"],
        histograms["background"],
        edges,
        signal_inclusive_yield=statistics["signal"]["inclusive_weight"],
        background_inclusive_yield=statistics["background"]["inclusive_weight"],
        signal_partition_weight=statistics["signal"]["partition_weight"],
        background_partition_weight=statistics["background"]["partition_weight"],
        target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
    )
    LAM_SIG = diagnostics["histogram_signal_yield"]
    LAM_BKG = diagnostics["histogram_background_yield"]
    np.savez(
        PRESEL_STATE_PATH,
        ratio_cut=PRESEL_RATIO_CUT,
        lambda_signal=LAM_SIG,
        lambda_background=LAM_BKG,
    )
    print(f"Saved PRESEL state to {PRESEL_STATE_PATH}")

print(f"PRESEL ratio cut: {PRESEL_RATIO_CUT:.6g}")
print(f"Post-selection yields: signal={LAM_SIG:.6g}, background={LAM_BKG:.6g}")
print(f"Post-selection B/S: {LAM_BKG / LAM_SIG:.3f}")


## Load the hybrid model from Exercise 5

We load the reference flow directly from its PyTorch checkpoint and each
member of the two ONNX density-ratio ensembles. Predictions are averaged as
ratios, exactly as in Exercise 5. Histogram calibration remains disabled.



In [ ]:
reference_flow = load_flow(
    "reference",
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    flow_type=REFERENCE_FLOW_TYPE,
    device=device,
    expected_features=FEATURES,
)

ratio_models = {}
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    ratio_models[sample_name] = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        scaler, model_proto = load_trained_model(
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        )
        ratio_models[sample_name].append(
            {"scaler": scaler, "model": as_inference_session(model_proto)}
        )


def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    values = np.asarray(values, dtype=np.float32)
    chunks = []
    for start in range(0, len(values), int(batch_size)):
        batch = pd.DataFrame(values[start : start + int(batch_size)], columns=FEATURES)
        member_predictions = []
        for pack in ratio_models[sample_name]:
            prediction = predict_with_model(
                batch,
                scaler=pack["scaler"],
                model=pack["model"],
            )
            member_predictions.append(
                np.asarray(prediction, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_predictions, axis=0), axis=0))
    ratio = np.concatenate(chunks) if chunks else np.empty(0, dtype=np.float64)
    if not np.isfinite(ratio).all():
        raise FloatingPointError(f"The {sample_name} ratio returned NaN or infinity.")
    return np.maximum(ratio, RATIO_FLOOR)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        n_generated += len(generated)
        n_passed += int(passes.sum())
        if np.any(passes):
            accepted_chunks.append(generated[passes])
            n_kept += int(passes.sum())
    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    return accepted.astype(np.float32, copy=False), n_passed / n_generated


def conditional_log_prob(flow_pack, values, acceptance, batch_size=65_536):
    return (
        np.asarray(
            flow_log_prob_x(flow_pack, values, batch_size=batch_size),
            dtype=np.float64,
        )
        - np.log(float(acceptance))
    )


print(f"Loaded reference flow: {reference_flow['path']}")
print("Loaded four signal/reference and four background/reference models.")



## Build the pilot target

Estimate $R_S$, $R_B$, $I_\mu$, $b_S(\mu)$, and $b_B(\mu)$ on the same pilot
sample as Exercise 6. Keep these coefficients fixed when evaluating
$A_{\mathrm{norm}}$ on new events. Apply the original clipping and floor only
to the proposal-training weights.


In [ ]:
def normalized_ratios(raw_signal, raw_background, weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
    signal_norm = float(np.sum(weights * raw_signal))
    background_norm = float(np.sum(weights * raw_background))
    return raw_signal / signal_norm, raw_background / background_norm


def fit_scan_normalization_target(raw_signal, raw_background, mu_values):
    """Estimate I_mu and both normalization coefficients on the pilot."""
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    ratio_normalization = np.asarray([raw_signal.mean(), raw_background.mean()])
    ratio_signal = raw_signal / ratio_normalization[0]
    ratio_background = raw_background / ratio_normalization[1]
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
    )
    mu_values = np.asarray(mu_values, dtype=np.float64)
    integrals = np.empty(len(mu_values))
    coefficients = np.empty((len(mu_values), 2))
    for k, mu in enumerate(mu_values):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        h_ratio = h_asimov / h_mu
        log_h_ratio = np.log(h_ratio)
        integrals[k] = np.mean(h_asimov * log_h_ratio)
        # b_s = b_s^A - b_s^T because the component shapes do not depend on mu.
        coefficients[k, 0] = np.mean(
            ratio_signal * LAM_SIG
            * (ASIMOV_MU_TRUE * (log_h_ratio + 1.0) - mu * h_ratio)
        )
        coefficients[k, 1] = np.mean(
            ratio_background * LAM_BKG * (log_h_ratio + 1.0 - h_ratio)
        )
    return {
        "ratio_normalization": ratio_normalization,
        "mu_values": mu_values,
        "integrals": integrals,
        "coefficients": coefficients,
        "scale": ASIMOV_MU_TRUE * LAM_SIG + LAM_BKG,
    }


def scan_influence_amplitude(raw_signal, raw_background, target):
    """Evaluate A_norm using fixed pilot normalizations and coefficients."""
    ratio_signal = (
        np.asarray(raw_signal, dtype=np.float64) / target["ratio_normalization"][0]
    )
    ratio_background = (
        np.asarray(raw_background, dtype=np.float64) / target["ratio_normalization"][1]
    )
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
    )
    amplitude_squared = np.zeros(len(ratio_signal), dtype=np.float64)
    for mu, integral, (b_signal, b_background) in zip(
        target["mu_values"], target["integrals"], target["coefficients"]
    ):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        y_mu = h_asimov * np.log(h_asimov / h_mu)
        influence_mu = (
            y_mu - integral
            - b_signal * (ratio_signal - 1.0)
            - b_background * (ratio_background - 1.0)
        )
        amplitude_squared += (influence_mu / target["scale"]) ** 2
    return np.sqrt(amplitude_squared / len(target["mu_values"]))


torch.manual_seed(SEED + 100)
pilot_values, REFERENCE_PRESEL_ACCEPTANCE = sample_preselected_flow(
    reference_flow,
    PILOT_EVENTS,
    batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
)
pilot_raw_signal = evaluate_ratio("signal", pilot_values)
pilot_raw_background = evaluate_ratio("background", pilot_values)
RATIO_NORMALIZATION = {
    "signal": float(np.mean(pilot_raw_signal)),
    "background": float(np.mean(pilot_raw_background)),
}

NIS_TARGET_STATE = fit_scan_normalization_target(
    pilot_raw_signal, pilot_raw_background, MU_DESIGN
)
pilot_amplitude = scan_influence_amplitude(
    pilot_raw_signal, pilot_raw_background, NIS_TARGET_STATE
)
np.savez(NIS_CACHE_DIR / "pilot_normalization_target.npz", **NIS_TARGET_STATE)
positive_amplitude = pilot_amplitude[pilot_amplitude > 0.0]
amplitude_floor = (
    NIS_TARGET_FLOOR_FRACTION * float(np.median(positive_amplitude))
)
amplitude_ceiling = float(
    np.quantile(pilot_amplitude, NIS_TARGET_CLIP_QUANTILE)
)
pilot_training_amplitude = np.clip(
    pilot_amplitude, amplitude_floor, amplitude_ceiling
)

print(f"Pilot reference events: {len(pilot_values):,}")
print(f"Reference PRESEL acceptance: {REFERENCE_PRESEL_ACCEPTANCE:.4%}")
print("Ratio normalizations:", RATIO_NORMALIZATION)
print(
    "Importance-amplitude quantiles:",
    np.quantile(pilot_amplitude, [0.0, 0.5, 0.9, 0.99, 0.999, 1.0]),
)
print(
    f"Training amplitude floor/ceiling: "
    f"{amplitude_floor:.3e}/{amplitude_ceiling:.3e}"
)


## Fit and tune the proposal

First fit the globally weighted likelihood. On independent reference events,
compare $J_{\mathrm{scan}}(\epsilon)=\mathbb E_{q_\phi}[(q_\phi/g_\epsilon)A_{\mathrm{norm}}^2]$
and $J_0(\epsilon)=\mathbb E_{q_\phi}[(q_\phi/g_\epsilon)(\Psi_0/\lambda_A)^2]$.

The optional second stage minimizes the unregularized scan objective.
It includes the derivative of the PRESEL normalization and takes one
full-sample optimizer step per epoch, accumulating gradients in memory-sized
batches. Its initial checkpoint is retained unless the tuning objective improves.
A separate validation sample and the repeated quadratures follow selection.


In [ ]:
torch.manual_seed(NIS_TRAIN_SEED)
importance_training_df = pd.DataFrame(pilot_values, columns=FEATURES)
importance_flow = train_flow(
    "asimov_importance", importance_training_df, features=FEATURES,
    model_dir=NIS_MODEL_DIR, model_config=NIS_MODEL_CONFIG,
    training_config=NIS_TRAINING_CONFIG, device=device,
    sample_weights=pilot_training_amplitude, max_train_events=None,
    load_if_available=True, seed=NIS_TRAIN_SEED,
)
del importance_training_df
gc.collect()

# Independent tuning bank: reused for all proposals and epsilon choices.
torch.manual_seed(TUNING_SEED)
tuning_values, _ = sample_preselected_flow(
    reference_flow, TUNING_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
tuning_raw_signal = evaluate_ratio("signal", tuning_values)
tuning_raw_background = evaluate_ratio("background", tuning_values)
tuning_amplitude = scan_influence_amplitude(
    tuning_raw_signal, tuning_raw_background, NIS_TARGET_STATE
)
q0_index = int(np.argmin(np.abs(NIS_TARGET_STATE["mu_values"])))
Q0_TARGET_STATE = {
    **NIS_TARGET_STATE,
    "mu_values": NIS_TARGET_STATE["mu_values"][q0_index:q0_index + 1],
    "integrals": NIS_TARGET_STATE["integrals"][q0_index:q0_index + 1],
    "coefficients": NIS_TARGET_STATE["coefficients"][q0_index:q0_index + 1],
}
tuning_q0_amplitude = scan_influence_amplitude(
    tuning_raw_signal, tuning_raw_background, Q0_TARGET_STATE
)
tuning_log_q = conditional_log_prob(
    reference_flow, tuning_values, REFERENCE_PRESEL_ACCEPTANCE
)


def tune_proposal(flow_pack, label, acceptance_seed):
    torch.manual_seed(acceptance_seed)
    probe, acceptance = sample_preselected_flow(
        flow_pack, ACCEPTANCE_CALIBRATION_EVENTS,
        batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
    )
    del probe
    log_ratio = conditional_log_prob(flow_pack, tuning_values, acceptance) - tuning_log_q
    table = epsilon_diagnostics(
        log_ratio, tuning_amplitude, tuning_q0_amplitude, DEFENSIVE_FRACTION_CANDIDATES
    )
    table["proposal"] = label
    table["acceptance"] = acceptance
    table["checkpoint"] = str(flow_pack["path"])
    return table, acceptance


mle_tuning, mle_acceptance = tune_proposal(importance_flow, "Global-weight MLE", TUNING_SEED + 1)
tuning_tables = [mle_tuning]
# Use the MLE scan-optimal epsilon during scan-variance fine-tuning.
fine_epsilon = float(mle_tuning.loc[mle_tuning["scan_objective"].idxmin(), "epsilon"])
if RUN_VARIANCE_FINETUNING and fine_epsilon < 1.0:
    pilot_log_q = conditional_log_prob(
        reference_flow, pilot_values, REFERENCE_PRESEL_ACCEPTANCE
    )
    importance_flow = finetune_variance(
        importance_flow, pilot_values, pilot_log_q, pilot_amplitude,
        tuning_values, tuning_log_q, tuning_amplitude,
        epsilon=fine_epsilon, model_dir=NIS_VARIANCE_MODEL_DIR,
        **VARIANCE_TRAINING_CONFIG,
    )
    current_tuning, IMPORTANCE_PRESEL_ACCEPTANCE = tune_proposal(
        importance_flow, "Variance fine-tuned", TUNING_SEED + 2
    )
    tuning_tables.append(current_tuning)
    # Confirm improvement using the same deployment-density diagnostic as MLE.
    metric = f"{EPSILON_OBJECTIVE}_objective"
    if current_tuning[metric].min() > mle_tuning[metric].min():
        importance_flow = load_flow(
            "asimov_importance", model_dir=NIS_MODEL_DIR,
            flow_type=NIS_MODEL_CONFIG["flow_type"], device=device,
            expected_features=FEATURES,
        )
        current_tuning = mle_tuning
        IMPORTANCE_PRESEL_ACCEPTANCE = mle_acceptance
        print("Retaining the MLE checkpoint: fine-tuning did not improve the tuning metric.")
    del pilot_log_q
else:
    current_tuning = mle_tuning
    IMPORTANCE_PRESEL_ACCEPTANCE = mle_acceptance

# Re-evaluate both old checkpoints with the SAME unclipped target and sample.
for index, (label, model_dir) in enumerate([
    ("Original Exercise 6", ORIGINAL_NIS_MODEL_DIR),
    ("Previous same-sample run", PREVIOUS_NIS_MODEL_DIR),
]):
    saved_path = checkpoint_path("asimov_importance", model_dir, NIS_MODEL_CONFIG["flow_type"])
    if saved_path.exists():
        saved_flow = load_flow(
            "asimov_importance", model_dir=model_dir,
            flow_type=NIS_MODEL_CONFIG["flow_type"], device=device,
            expected_features=FEATURES,
        )
        saved_table, _ = tune_proposal(saved_flow, label, TUNING_SEED + 10 + index)
        tuning_tables.append(saved_table)
        del saved_flow
        gc.collect()

proposal_tuning = pd.concat(tuning_tables, ignore_index=True)
proposal_tuning.to_csv(NIS_CACHE_DIR / "proposal_epsilon_tuning.csv", index=False)
display(proposal_tuning[[
    "proposal", "epsilon", "predicted_scan_gain", "predicted_q0_gain", "acceptance"
]])
selected = current_tuning.loc[current_tuning[f"{EPSILON_OBJECTIVE}_objective"].idxmin()]
DEFENSIVE_REFERENCE_FRACTION = float(selected["epsilon"])
print(f"Selected epsilon={DEFENSIVE_REFERENCE_FRACTION:g} using the {EPSILON_OBJECTIVE} tuning objective.")
print(f"Deployed proposal: {importance_flow['path']}")
print(f"Importance-flow PRESEL acceptance: {IMPORTANCE_PRESEL_ACCEPTANCE:.4%}")

del (
    pilot_values, pilot_raw_signal, pilot_raw_background, pilot_amplitude,
    pilot_training_amplitude, positive_amplitude, tuning_values,
    tuning_raw_signal, tuning_raw_background, tuning_amplitude,
    tuning_q0_amplitude, tuning_log_q,
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()



## Independent validation

Compare the selected proposal with $A_{\mathrm{norm}}$ on fresh reference events.
Report both variance proxies using the unclipped amplitude. The selected
$\epsilon$ stays fixed; these events do not select the checkpoint or mixture.


In [ ]:
VALIDATION_EVENTS = 200_000
torch.manual_seed(STUDY_SEED + 400)
validation_values, _ = sample_preselected_flow(
    reference_flow, VALIDATION_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
validation_raw_signal = evaluate_ratio("signal", validation_values)
validation_raw_background = evaluate_ratio("background", validation_values)
validation_amplitude = scan_influence_amplitude(
    validation_raw_signal, validation_raw_background, NIS_TARGET_STATE
)
validation_training_amplitude = np.clip(
    validation_amplitude, amplitude_floor, np.inf
)

validation_log_q = conditional_log_prob(
    reference_flow, validation_values, REFERENCE_PRESEL_ACCEPTANCE
)
validation_log_g = conditional_log_prob(
    importance_flow, validation_values, IMPORTANCE_PRESEL_ACCEPTANCE
)
log_target = np.log(validation_training_amplitude)
log_proposal_ratio = validation_log_g - validation_log_q
log_target_centered = log_target - np.mean(log_target)
log_proposal_centered = log_proposal_ratio - np.mean(log_proposal_ratio)
proposal_target_correlation = float(
    np.corrcoef(log_target_centered, log_proposal_centered)[0, 1]
)
proposal_target_slope = float(
    np.polyfit(log_target_centered, log_proposal_centered, deg=1)[0]
)
proposal_target_rmse = float(
    np.sqrt(np.mean((log_proposal_centered - log_target_centered) ** 2))
)

validation_q0_amplitude = scan_influence_amplitude(
    validation_raw_signal, validation_raw_background, Q0_TARGET_STATE
)
defensive_diagnostics = epsilon_diagnostics(
    log_proposal_ratio, validation_amplitude, validation_q0_amplitude,
    DEFENSIVE_FRACTION_CANDIDATES,
)

rng = np.random.default_rng(STUDY_SEED + 401)
plot_indices = rng.choice(
    len(validation_values), size=min(40_000, len(validation_values)), replace=False
)
limit = np.quantile(
    np.abs(
        np.concatenate(
            [log_target_centered[plot_indices], log_proposal_centered[plot_indices]]
        )
    ),
    0.995,
)

fig, ax = plt.subplots(figsize=(6.4, 5.5))
hexbin = ax.hexbin(
    log_target_centered[plot_indices],
    log_proposal_centered[plot_indices],
    gridsize=70,
    bins="log",
    mincnt=1,
    cmap="viridis",
)
ax.plot([-limit, limit], [-limit, limit], color="black", ls="--", lw=1.5)
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_xlabel(r"Centered $\log A_{\mathrm{norm}}(\mathbf{x})$")
ax.set_ylabel(r"Centered $\log[g_{\boldsymbol{\eta}}(\mathbf{x})/q_\phi(\mathbf{x})]$")
ax.set_title("Neural proposal versus variance-optimal target")
ax.text(
    0.04,
    0.96,
    rf"$\rho={proposal_target_correlation:.3f}$"
    + "\n"
    + rf"slope$={proposal_target_slope:.3f}$"
    + "\n"
    + rf"RMS$={proposal_target_rmse:.3f}$",
    transform=ax.transAxes,
    va="top",
)
fig.colorbar(hexbin, ax=ax, label="log count")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "proposal_target_closure.png", dpi=160)
export_exercise6_figure(fig, "proposal_target_closure")
plt.show()

print(f"Proposal/target correlation: {proposal_target_correlation:.6f}")
print(f"Proposal/target slope: {proposal_target_slope:.6f}")
print(f"Centered log-ratio RMS: {proposal_target_rmse:.6f}")
display(defensive_diagnostics)
selected_row = defensive_diagnostics.loc[
    np.isclose(
        defensive_diagnostics["epsilon"], DEFENSIVE_REFERENCE_FRACTION
    )
].iloc[0]
print(
    f"Using epsilon={DEFENSIVE_REFERENCE_FRACTION:g}; held-out predicted "
    f"scan gain={selected_row['predicted_scan_gain']:.3f}, "
    f"q0 gain={selected_row['predicted_q0_gain']:.3f}."
)


defensive_diagnostics.to_csv(NIS_CACHE_DIR / "defensive_diagnostics.csv", index=False)

np.savez_compressed(
    NIS_CACHE_DIR / "validation_variance_inputs.npz",
    scan_amplitude=validation_amplitude, q0_amplitude=validation_q0_amplitude,
    log_g_over_q=log_proposal_ratio,
)


## Defensive sampling

Use the selected $g_\epsilon=(1-\epsilon)g_{\boldsymbol{\eta}}+\epsilon q_\phi$.
The process ratios are normalized again on every deployment quadrature.
Keep all events, including influential tail events.


In [ ]:

def log_defensive_proposal(values):
    log_q = conditional_log_prob(
        reference_flow, values, REFERENCE_PRESEL_ACCEPTANCE
    )
    log_g = conditional_log_prob(
        importance_flow, values, IMPORTANCE_PRESEL_ACCEPTANCE
    )
    log_mix = log_q - mixture_log_weights(log_g - log_q, DEFENSIVE_REFERENCE_FRACTION)
    return log_mix, log_q


def sample_defensive_proposal(n_events, seed):
    rng = np.random.default_rng(seed)
    n_reference = int(rng.binomial(int(n_events), DEFENSIVE_REFERENCE_FRACTION))
    n_importance = int(n_events) - n_reference

    torch.manual_seed(seed + 1)
    reference_values, _ = sample_preselected_flow(
        reference_flow, n_reference, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
    ) if n_reference else (np.empty((0, N_DIM), dtype=np.float32), np.nan)
    torch.manual_seed(seed + 2)
    importance_values, _ = sample_preselected_flow(
        importance_flow, n_importance, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
    ) if n_importance else (np.empty((0, N_DIM), dtype=np.float32), np.nan)

    values = np.concatenate([reference_values, importance_values], axis=0)
    values = values[rng.permutation(len(values))]
    log_mix, log_q = log_defensive_proposal(values)
    return values, log_q - log_mix


def normalized_importance_weights(log_weights):
    log_weights = np.asarray(log_weights, dtype=np.float64)
    return np.exp(log_weights - logsumexp(log_weights))


torch.manual_seed(STUDY_SEED + 500)
proposal_validation_values, proposal_validation_log_weights = (
    sample_defensive_proposal(VALIDATION_EVENTS, STUDY_SEED + 500)
)
proposal_validation_weights = normalized_importance_weights(
    proposal_validation_log_weights
)
proposal_validation_ess = 1.0 / np.sum(proposal_validation_weights**2)

fig, axes = plt.subplots(1, N_DIM, figsize=(3.2 * N_DIM, 3.2))
for axis, feature_index, feature_name in zip(axes, range(N_DIM), FEATURES):
    combined = np.concatenate(
        [validation_values[:, feature_index], proposal_validation_values[:, feature_index]]
    )
    edges = np.quantile(combined, np.linspace(0.001, 0.999, 41))
    edges = np.unique(edges)
    axis.hist(
        validation_values[:, feature_index],
        bins=edges,
        density=True,
        histtype="step",
        lw=2,
        label=r"Direct $q_\phi$",
    )
    axis.hist(
        proposal_validation_values[:, feature_index],
        bins=edges,
        weights=proposal_validation_weights,
        density=True,
        histtype="step",
        lw=1.8,
        label=r"$g_\epsilon$, weighted",
    )
    axis.set_xlabel(feature_name)
    axis.set_ylabel("Density")
axes[0].legend(fontsize=8)
fig.suptitle("Importance-reweighted reference closure", y=1.02)
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "importance_reweighted_reference_closure.png", dpi=160)
export_exercise6_figure(fig, "importance_reweighted_reference_closure")
plt.show()

print(
    f"Defensive-proposal ESS: {proposal_validation_ess:,.0f}/"
    f"{VALIDATION_EVENTS:,} ({proposal_validation_ess / VALIDATION_EVENTS:.1%})"
)
print(
    "Raw q/g weight quantiles:",
    np.quantile(
        np.exp(proposal_validation_log_weights),
        [0.0, 0.5, 0.9, 0.99, 0.999, 1.0],
    ),
)

del (
    validation_values,
    validation_raw_signal,
    validation_raw_background,
    validation_amplitude,
    validation_q0_amplitude,
    validation_training_amplitude,
    validation_log_q,
    validation_log_g,
    proposal_validation_values,
    proposal_validation_log_weights,
    proposal_validation_weights,
)
gc.collect()


## Weighted Asimov scan on a general quadrature

Every finite quadrature uses weights $\omega_m$ normalized to one. Before
constructing the scan, each learned process ratio is normalized on that same
quadrature,

$$
\widetilde r_j(x_m)=
\frac{\widehat r_j(x_m)}{\sum_n\omega_n\widehat r_j(x_n)}.
$$

Consequently, the Asimov score at $\mu_A=1$ vanishes exactly for both direct
and importance sampling. The comparison below therefore probes the difficult
part of the calculation—the curvature and finite displacement of the scan—
rather than a trivial shift of its minimum.



In [ ]:
def asimov_scan(raw_signal, raw_background, mu_values, log_weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if log_weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = normalized_importance_weights(log_weights)

    ratio_signal, ratio_background = normalized_ratios(
        raw_signal, raw_background, weights
    )
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal
        + LAM_BKG * ratio_background
    )

    scan = []
    for mu in np.asarray(mu_values, dtype=np.float64):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        integral = np.sum(weights * h_asimov * np.log(h_asimov / h_mu))
        statistic = 2.0 * (
            (mu - ASIMOV_MU_TRUE) * LAM_SIG + integral
        )
        scan.append(max(0.0, float(statistic)))

    score_at_truth = 2.0 * LAM_SIG * (
        1.0 - float(np.sum(weights * ratio_signal))
    )
    ess = 1.0 / np.sum(weights**2)
    return np.asarray(scan), float(score_at_truth), float(ess)


def evaluate_hybrid_ratios(values):
    return evaluate_ratio("signal", values), evaluate_ratio("background", values)


print("Constructing a high-statistics numerical benchmark...")
torch.manual_seed(SEED + 600)
benchmark_values, _ = sample_preselected_flow(
    reference_flow, BENCHMARK_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
benchmark_raw_signal, benchmark_raw_background = evaluate_hybrid_ratios(
    benchmark_values
)
del benchmark_values
gc.collect()
BENCHMARK_SCAN, BENCHMARK_SCORE, BENCHMARK_ESS = asimov_scan(
    benchmark_raw_signal, benchmark_raw_background, MU_SCAN
)
zero_index = int(np.argmin(np.abs(MU_SCAN)))
BENCHMARK_Q_ZERO = float(BENCHMARK_SCAN[zero_index])
BENCHMARK_SIGMA = ASIMOV_MU_TRUE / np.sqrt(BENCHMARK_Q_ZERO)

# A block estimate makes clear that this is a precise numerical benchmark, not
# an analytic truth curve.
benchmark_block_q0 = []
for block in np.array_split(np.arange(BENCHMARK_EVENTS), BENCHMARK_BLOCKS):
    block_scan, _, _ = asimov_scan(
        benchmark_raw_signal[block], benchmark_raw_background[block], MU_SCAN
    )
    benchmark_block_q0.append(block_scan[zero_index])
BENCHMARK_Q_ZERO_SE = float(
    np.std(benchmark_block_q0, ddof=1) / np.sqrt(BENCHMARK_BLOCKS)
)

print(f"Benchmark q_0,A: {BENCHMARK_Q_ZERO:.8f} +/- {BENCHMARK_Q_ZERO_SE:.8f} (blocks)")
print(f"Benchmark sigma_mu: {BENCHMARK_SIGMA:.8f}")
print(f"Benchmark score at mu_A: {BENCHMARK_SCORE:+.3e}")

fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.plot(MU_SCAN, BENCHMARK_SCAN, color="black", lw=2.4)
ax.axvline(ASIMOV_MU_TRUE, color="0.5", ls="--", lw=1.2)
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$t_A(\mu)$")
ax.set_title("High-statistics hybrid-model Asimov benchmark")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "asimov_benchmark_scan.png", dpi=160)
export_exercise6_figure(fig, "asimov_benchmark_scan")
plt.show()



## Repeated quadratures and epsilon comparison

Use fresh reference and proposal pools with shared mixture uniforms. Record
the selected mixture and direct reference at every original sample size.
At $M=2{,}048$, test every $\epsilon$ on these same pools, retaining every
repetition. The tuning choice remains fixed during this independent study.

Save the largest importance weights and component-normalizer deviations to
help diagnose influential repetitions.


In [ ]:
maximum_study_size = int(np.max(STUDY_SAMPLE_SIZES))
rows = []
epsilon_rows = []
showcase_scans = {"Direct reference": [], "Neural importance": []}


def quadrature_pool(values):
    raw_signal, raw_background = evaluate_hybrid_ratios(values)
    log_q = conditional_log_prob(reference_flow, values, REFERENCE_PRESEL_ACCEPTANCE)
    log_g = conditional_log_prob(importance_flow, values, IMPORTANCE_PRESEL_ACCEPTANCE)
    return {"signal": raw_signal, "background": raw_background,
            "log_g_over_q": log_g - log_q}


for repetition in range(N_REPETITIONS):
    torch.manual_seed(STUDY_SEED + 10_000 + repetition)
    direct_values, _ = sample_preselected_flow(
        reference_flow, maximum_study_size,
        batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
    )
    reference_pool = quadrature_pool(direct_values)
    del direct_values
    torch.manual_seed(STUDY_SEED + 20_000 + repetition)
    proposal_values, _ = sample_preselected_flow(
        importance_flow, maximum_study_size,
        batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
    )
    proposal_pool = quadrature_pool(proposal_values)
    del proposal_values
    uniforms = np.random.default_rng(STUDY_SEED + 50_000 + repetition).random(maximum_study_size)
    direct_raw_signal = reference_pool["signal"]
    direct_raw_background = reference_pool["background"]
    importance_raw_signal, importance_raw_background, importance_log_weights = mix_quadratures(
        reference_pool, proposal_pool, uniforms, DEFENSIVE_REFERENCE_FRACTION
    )

    # Paired epsilon comparison at the original showcase size; no NN re-evaluation.
    for epsilon in DEFENSIVE_FRACTION_CANDIDATES:
        raw_s, raw_b, log_w = mix_quadratures(
            reference_pool, proposal_pool, uniforms, epsilon
        )
        scan, _, _ = asimov_scan(
            raw_s[:SHOWCASE_SAMPLE_SIZE], raw_b[:SHOWCASE_SAMPLE_SIZE],
            MU_SCAN, log_weights=log_w[:SHOWCASE_SAMPLE_SIZE],
        )
        epsilon_rows.append({
            "epsilon": epsilon, "repetition": repetition,
            "sample_size": SHOWCASE_SAMPLE_SIZE,
            "q_zero": float(scan[zero_index]),
            "q_zero_error": float(scan[zero_index] - BENCHMARK_Q_ZERO),
            "scan_rmse": float(np.sqrt(np.mean((scan - BENCHMARK_SCAN)**2))),
        })

    for sample_size in STUDY_SAMPLE_SIZES:
        sample_size = int(sample_size)
        for method, raw_signal, raw_background, log_weights in [
            ("Direct reference", direct_raw_signal[:sample_size],
             direct_raw_background[:sample_size], None),
            ("Neural importance", importance_raw_signal[:sample_size],
             importance_raw_background[:sample_size], importance_log_weights[:sample_size]),
        ]:
            scan, score, ess = asimov_scan(
                raw_signal, raw_background, MU_SCAN, log_weights=log_weights
            )
            q_zero = float(scan[zero_index])
            weights = (np.full(sample_size, 1.0 / sample_size)
                       if log_weights is None else normalized_importance_weights(log_weights))
            rows.append({
                "method": method, "repetition": repetition, "sample_size": sample_size,
                "q_zero": q_zero, "q_zero_error": q_zero - BENCHMARK_Q_ZERO,
                "sigma_mu": ASIMOV_MU_TRUE / np.sqrt(q_zero),
                "scan_rmse": float(np.sqrt(np.mean((scan - BENCHMARK_SCAN)**2))),
                "score_at_truth": score, "ess": ess,
                "max_importance_weight": (1.0 if log_weights is None
                                          else float(np.exp(np.max(log_weights)))),
                "max_quadrature_weight": float(np.max(weights)),
                "signal_normalizer_over_pilot": float(
                    np.sum(weights * raw_signal) / NIS_TARGET_STATE["ratio_normalization"][0]
                ),
                "background_normalizer_over_pilot": float(
                    np.sum(weights * raw_background) / NIS_TARGET_STATE["ratio_normalization"][1]
                ),
            })
            if sample_size == SHOWCASE_SAMPLE_SIZE and repetition < 8:
                showcase_scans[method].append(scan)
    if (repetition + 1) % max(1, N_REPETITIONS // 8) == 0:
        print(f"Completed {repetition + 1}/{N_REPETITIONS} repetitions", flush=True)

study_results = pd.DataFrame(rows)


def summarize_group(group):
    return pd.Series(
        {
            "q0_mean": group["q_zero"].mean(),
            "q0_bias": group["q_zero_error"].mean(),
            "q0_std": group["q_zero"].std(ddof=1),
            "q0_rmse": np.sqrt(np.mean(group["q_zero_error"] ** 2)),
            "scan_rmse": np.sqrt(np.mean(group["scan_rmse"] ** 2)),
            "sigma_rmse": np.sqrt(
                np.mean((group["sigma_mu"] - BENCHMARK_SIGMA) ** 2)
            ),
            "mean_ess": group["ess"].mean(),
            "max_abs_score": np.max(np.abs(group["score_at_truth"])),
        }
    )


summary_rows = []
for (method, sample_size), group in study_results.groupby(
    ["method", "sample_size"], sort=False
):
    row = summarize_group(group).to_dict()
    row.update({"method": method, "sample_size": int(sample_size)})
    summary_rows.append(row)
study_summary = pd.DataFrame(summary_rows)

direct_variance = (
    study_summary.loc[study_summary["method"] == "Direct reference"]
    .set_index("sample_size")["q0_std"] ** 2
)
importance_rows = study_summary[study_summary["method"] == "Neural importance"].copy()
importance_rows["variance_reduction"] = importance_rows["sample_size"].map(
    direct_variance
) / importance_rows["q0_std"] ** 2
study_summary = study_summary.merge(
    importance_rows[["sample_size", "variance_reduction"]],
    on="sample_size",
    how="left",
)

display(
    study_summary[
        [
            "method",
            "sample_size",
            "q0_bias",
            "q0_std",
            "q0_rmse",
            "scan_rmse",
            "mean_ess",
            "max_abs_score",
            "variance_reduction",
        ]
    ]
)


study_results.to_csv(NIS_CACHE_DIR / "study_results.csv", index=False)
study_summary.to_csv(NIS_CACHE_DIR / "study_summary.csv", index=False)
np.savez(
    NIS_CACHE_DIR / "benchmark_and_scans.npz",
    mu_scan=MU_SCAN,
    benchmark_scan=BENCHMARK_SCAN,
    benchmark_q0=BENCHMARK_Q_ZERO,
    benchmark_q0_se=BENCHMARK_Q_ZERO_SE,
    benchmark_block_q0=np.asarray(benchmark_block_q0),
    showcase_direct=np.asarray(showcase_scans["Direct reference"]),
    showcase_nis=np.asarray(showcase_scans["Neural importance"]),
)

import json

comparison_settings = {
    "target": "same_sample_normalization",
    "seed": SEED,
    "training_seed": NIS_TRAIN_SEED,
    "study_seed": STUDY_SEED,
    "tuning_seed": TUNING_SEED,
    "tuning_events": TUNING_EVENTS,
    "epsilon_objective": EPSILON_OBJECTIVE,
    "epsilon_candidates": list(DEFENSIVE_FRACTION_CANDIDATES),
    "variance_finetuning": RUN_VARIANCE_FINETUNING,
    "variance_training_config": VARIANCE_TRAINING_CONFIG,
    "mu_true": ASIMOV_MU_TRUE,
    "mu_design": MU_DESIGN.tolist(),
    "mu_scan": MU_SCAN.tolist(),
    "pilot_events": PILOT_EVENTS,
    "validation_events": VALIDATION_EVENTS,
    "acceptance_calibration_events": ACCEPTANCE_CALIBRATION_EVENTS,
    "benchmark_events": BENCHMARK_EVENTS,
    "benchmark_blocks": BENCHMARK_BLOCKS,
    "study_sample_sizes": STUDY_SAMPLE_SIZES.tolist(),
    "repetitions": N_REPETITIONS,
    "showcase_sample_size": SHOWCASE_SAMPLE_SIZE,
    "defensive_reference_fraction": DEFENSIVE_REFERENCE_FRACTION,
    "target_clip_quantile": NIS_TARGET_CLIP_QUANTILE,
    "target_floor_fraction": NIS_TARGET_FLOOR_FRACTION,
    "preselection_ratio_cut": PRESEL_RATIO_CUT,
    "lambda_signal": LAM_SIG,
    "lambda_background": LAM_BKG,
    "reference_preselection_acceptance": REFERENCE_PRESEL_ACCEPTANCE,
    "importance_preselection_acceptance": IMPORTANCE_PRESEL_ACCEPTANCE,
    "reference_checkpoint": str(reference_flow["path"]),
    "importance_checkpoint": str(importance_flow["path"]),
    "ratio_model_dirs": {key: str(value) for key, value in RATIO_MODEL_DIR.items()},
    "model_config": NIS_MODEL_CONFIG,
    "training_config": NIS_TRAINING_CONFIG,
}
(NIS_CACHE_DIR / "comparison_settings.json").write_text(
    json.dumps(comparison_settings, indent=2) + "\n"
)

epsilon_study = pd.DataFrame(epsilon_rows)
epsilon_summary_rows = []
showcase_direct = study_results[
    (study_results["method"] == "Direct reference")
    & (study_results["sample_size"] == SHOWCASE_SAMPLE_SIZE)
]
for epsilon, group in epsilon_study.groupby("epsilon", sort=True):
    epsilon_summary_rows.append({
        "epsilon": epsilon,
        "q0_std": group["q_zero"].std(ddof=1),
        "q0_rmse": np.sqrt(np.mean(group["q_zero_error"]**2)),
        "scan_rmse": np.sqrt(np.mean(group["scan_rmse"]**2)),
        "variance_reduction": showcase_direct["q_zero"].var(ddof=1)
                              / group["q_zero"].var(ddof=1),
        "selected_on_tuning": bool(np.isclose(epsilon, DEFENSIVE_REFERENCE_FRACTION)),
    })
epsilon_summary = pd.DataFrame(epsilon_summary_rows)
epsilon_study.to_csv(NIS_CACHE_DIR / "epsilon_study_results.csv", index=False)
epsilon_summary.to_csv(NIS_CACHE_DIR / "epsilon_study_summary.csv", index=False)
print(f"Independent epsilon comparison at M={SHOWCASE_SAMPLE_SIZE:,}:")
display(epsilon_summary)


In [ ]:
colors = {"Direct reference": "C3", "Neural importance": "C0"}
markers = {"Direct reference": "o", "Neural importance": "s"}

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for method, group in study_summary.groupby("method", sort=False):
    group = group.sort_values("sample_size")
    axes[0].plot(
        group["sample_size"],
        group["q0_rmse"],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )
    axes[1].plot(
        group["sample_size"],
        group["scan_rmse"],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )

nis_summary = study_summary[
    study_summary["method"] == "Neural importance"
].sort_values("sample_size")
axes[2].plot(
    nis_summary["sample_size"],
    nis_summary["variance_reduction"],
    marker="D",
    color="C2",
    lw=2,
)
axes[2].axhline(1.0, color="black", ls="--", lw=1.2)

for axis in axes[:2]:
    axis.set_xscale("log", base=2)
    axis.set_yscale("log")
    axis.grid(alpha=0.25)
    axis.legend()
axes[2].set_xscale("log", base=2)
axes[2].grid(alpha=0.25)
axes[0].set_xlabel("Number of Asimov points")
axes[0].set_ylabel(r"RMSE of $q_{0,A}$")
axes[0].set_title("Discovery statistic")
axes[1].set_xlabel("Number of Asimov points")
axes[1].set_ylabel(r"RMS error over $t_A(\mu)$")
axes[1].set_title("Complete likelihood scan")
axes[2].set_xlabel("Number of Asimov points")
axes[2].set_ylabel(r"$\mathrm{Var}_{q_\phi}/\mathrm{Var}_{\rm NIS}$")
axes[2].set_title("Approximate event-saving factor")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "nis_asimov_convergence.png", dpi=160)
export_exercise6_figure(fig, "nis_asimov_convergence")
plt.show()



In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 5.0))
for method in ["Direct reference", "Neural importance"]:
    for index, scan in enumerate(showcase_scans[method]):
        ax.plot(
            MU_SCAN,
            scan,
            color=colors[method],
            alpha=0.20,
            lw=1.2,
            label=method if index == 0 else None,
        )
ax.plot(MU_SCAN, BENCHMARK_SCAN, color="black", lw=2.6, label="Benchmark")
ax.axvline(ASIMOV_MU_TRUE, color="0.5", ls="--", lw=1.0)
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$t_A(\mu)$")
ax.set_title(f"Repeated {SHOWCASE_SAMPLE_SIZE:,}-point Asimov scans")
ax.legend()
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "nis_repeated_small_asimov_scans.png", dpi=160)
export_exercise6_figure(fig, "nis_repeated_small_asimov_scans")
plt.show()



In [ ]:
selected_results = study_results[
    study_results["sample_size"] == SHOWCASE_SAMPLE_SIZE
]
fig, ax = plt.subplots(figsize=(6.5, 4.8))
data = [
    selected_results.loc[selected_results["method"] == method, "q_zero"].to_numpy()
    for method in ["Direct reference", "Neural importance"]
]
box = ax.boxplot(
    data,
    labels=["Direct reference", "Neural importance"],
    patch_artist=True,
    showmeans=True,
)
for patch, color in zip(box["boxes"], [colors["Direct reference"], colors["Neural importance"]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)
ax.axhline(BENCHMARK_Q_ZERO, color="black", lw=2, label="Benchmark")
ax.set_ylabel(r"$q_{0,A}$")
ax.set_title(f"Equal-cost comparison with {SHOWCASE_SAMPLE_SIZE:,} points")
ax.legend()
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "nis_q0_equal_cost.png", dpi=160)
export_exercise6_figure(fig, "nis_q0_equal_cost")
plt.show()



## Interpreting the result

A convincing proof of principle has several simultaneous features:

1. **Proposal closure:** $\log(g_{\boldsymbol{\eta}}/q_\phi)$ follows $\log A_{\mathrm{norm}}$ on held-out
   reference events.
2. **Importance closure:** proposal events reweighted by $q_\phi/g_\epsilon$
   reproduce the reference feature distributions.
3. **No visible bias:** the mean neural-importance estimate remains compatible
   with the high-statistics benchmark as $M$ increases.
4. **Reduced numerical variance:** the $q_{0,A}$ and complete-scan RMSE are
   smaller than direct reference sampling at equal $M$.
5. **Exact Asimov minimum:** the score at $\mu_A=1$ remains zero up to floating
   point precision because the ratios are normalized on each weighted
   quadrature.

The variance-reduction panel translates the gain into an intuitive number. A
value of ten means that, for estimating $q_{0,A}$, approximately ten times as
many ordinary reference points would be required to match the variance of the
neural-importance construction.

This comparison is conditional on the trained Exercise 5 hybrid model. Drawing
more proposal events reduces quadrature uncertainty, but it does not reduce
modeling uncertainty in the reference flow or density-ratio ensembles.



The current notebook formulas use the same population target. The tuning table
compares the old checkpoints, the corrected likelihood fit, and variance
fine-tuning on common events. The independent epsilon table isolates the
defensive-mixture choice. Keep all repetitions when comparing variances.


In [ ]:
showcase_summary = study_summary[
    study_summary["sample_size"] == SHOWCASE_SAMPLE_SIZE
].set_index("method")
showcase_gain = float(
    showcase_summary.loc["Neural importance", "variance_reduction"]
)
showcase_direct_rmse = float(showcase_summary.loc["Direct reference", "q0_rmse"])
showcase_nis_rmse = float(showcase_summary.loc["Neural importance", "q0_rmse"])
maximum_score = float(study_summary["max_abs_score"].max())

print(f"At M={SHOWCASE_SAMPLE_SIZE:,}:")
print(f"  direct q0 RMSE = {showcase_direct_rmse:.6g}")
print(f"  NIS q0 RMSE    = {showcase_nis_rmse:.6g}")
print(f"  variance-reduction factor = {showcase_gain:.3f}")
print(f"Maximum |Asimov score at mu_A| over study = {maximum_score:.3e}")

if showcase_gain > 1.0:
    print(
        "Proof-of-principle result: neural importance sampling reduces the "
        "equal-cost variance for q_0,A."
    )
else:
    print(
        "The proposal is not yet more efficient at the showcase size. Inspect "
        "proposal closure and importance-weight tails before increasing its capacity."
    )



## Matched-precision computational benchmark

The variance study above provides a statistically controlled pair for a timing
comparison. We keep the neural-importance sample size fixed at
$M_{\rm NIS}=2{,}048$ and select the direct-reference sample size whose measured
standard deviation of $q_{0,A}$ is closest. The complete-scan RMSE is reported
as a second check that the two quadratures have comparable precision.

We time three scopes:

1. the complete 61-point $t_A(\mu)$ scan;
2. a bounded scalar minimization of the same, **unclipped** Asimov objective; and
3. quadrature construction followed by the minimization.

The third scope includes event generation, PRESEL, the frozen signal/reference
and background/reference ratio ensembles, and—for neural importance
sampling—the reference/proposal density evaluations needed for the exact
importance weights. It deliberately excludes construction and training of the
neural proposal, which is a one-time cost that can be amortized over many
likelihood evaluations. This benchmark calls only already-loaded models and
does not retrain anything.

Because ratio normalization makes $\mu_A=1$ stationary on every finite
quadrature, the fitted $\widehat{\mu}$ itself has essentially no meaningful
finite-sample variance here. “Matched precision” therefore refers to the
measured variance of $q_{0,A}$ and the RMSE of the complete likelihood scan.
The wall times are hardware-dependent and should always be quoted together
with the sample sizes and precision columns below.


In [ ]:
from scipy.optimize import minimize_scalar
from time import perf_counter

# The fit-only measurements are inexpensive, so repeat them enough times to
# obtain stable medians. End-to-end repetitions include neural-network and
# flow inference and are consequently kept smaller.
FIT_TIMING_REPETITIONS = 200
FIT_TIMING_WARMUPS = 5
END_TO_END_TIMING_REPETITIONS = 8
END_TO_END_TIMING_WARMUPS = 1
MINIMIZER_BOUNDS = (float(np.min(MU_SCAN)), float(np.max(MU_SCAN)))
MINIMIZER_XATOL = 1.0e-7

# Match the direct quadrature to the NIS showcase point using the empirically
# measured q0 standard deviation from the independent repeated-quadrature study.
nis_precision_row = study_summary[
    (study_summary["method"] == "Neural importance")
    & (study_summary["sample_size"] == SHOWCASE_SAMPLE_SIZE)
].iloc[0]

direct_precision_candidates = study_summary[
    study_summary["method"] == "Direct reference"
].copy()
direct_precision_candidates["match_distance"] = np.abs(
    np.log(
        direct_precision_candidates["q0_std"]
        / float(nis_precision_row["q0_std"])
    )
)
direct_precision_row = direct_precision_candidates.sort_values(
    "match_distance"
).iloc[0]

MATCHED_NIS_SIZE = int(nis_precision_row["sample_size"])
MATCHED_DIRECT_SIZE = int(direct_precision_row["sample_size"])


def prepare_asimov_objective(raw_signal, raw_background, log_weights=None):
    """Return the un-clipped finite-quadrature Asimov objective."""
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if log_weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = normalized_importance_weights(log_weights)

    ratio_signal, ratio_background = normalized_ratios(
        raw_signal, raw_background, weights
    )
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal
        + LAM_BKG * ratio_background
    )

    def objective(mu):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        integral = np.sum(weights * h_asimov * np.log(h_asimov / h_mu))
        return float(
            2.0 * ((mu - ASIMOV_MU_TRUE) * LAM_SIG + integral)
        )

    return objective


def minimize_asimov(raw_signal, raw_background, log_weights=None):
    objective = prepare_asimov_objective(
        raw_signal, raw_background, log_weights=log_weights
    )
    return minimize_scalar(
        objective,
        bounds=MINIMIZER_BOUNDS,
        method="bounded",
        options={"xatol": MINIMIZER_XATOL},
    )


def synchronize_accelerator():
    # The fit-only calculation is NumPy/CPU. This synchronization is needed
    # for the end-to-end measurements when the flows run on CUDA.
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def time_repeated(call, repetitions, warmups):
    for _ in range(int(warmups)):
        call()
    durations = []
    outputs = []
    for _ in range(int(repetitions)):
        synchronize_accelerator()
        start = perf_counter()
        output = call()
        synchronize_accelerator()
        durations.append(perf_counter() - start)
        outputs.append(output)
    return np.asarray(durations, dtype=np.float64), outputs


def summarize_durations(durations):
    q25, median, q75 = np.percentile(
        1.0e3 * np.asarray(durations), [25.0, 50.0, 75.0]
    )
    return {
        "q25_ms": float(q25),
        "median_ms": float(median),
        "q75_ms": float(q75),
    }


def construct_quadrature_and_minimize(method, n_events, seed):
    """Time-relevant work only: no model construction or training."""
    if method == "Direct reference":
        torch.manual_seed(seed)
        values, _ = sample_preselected_flow(
            reference_flow,
            n_events,
            batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
        )
        raw_signal, raw_background = evaluate_hybrid_ratios(values)
        log_weights = None
    elif method == "Neural importance":
        values, log_weights = sample_defensive_proposal(n_events, seed)
        raw_signal, raw_background = evaluate_hybrid_ratios(values)
    else:
        raise ValueError(f"Unknown timing method: {method}")

    return minimize_asimov(
        raw_signal, raw_background, log_weights=log_weights
    )


timing_inputs = {
    "Direct reference": {
        "sample_size": MATCHED_DIRECT_SIZE,
        "raw_signal": direct_raw_signal[:MATCHED_DIRECT_SIZE],
        "raw_background": direct_raw_background[:MATCHED_DIRECT_SIZE],
        "log_weights": None,
        "precision": direct_precision_row,
    },
    "Neural importance": {
        "sample_size": MATCHED_NIS_SIZE,
        "raw_signal": importance_raw_signal[:MATCHED_NIS_SIZE],
        "raw_background": importance_raw_background[:MATCHED_NIS_SIZE],
        "log_weights": importance_log_weights[:MATCHED_NIS_SIZE],
        "precision": nis_precision_row,
    },
}

timing_rows = []
for method, inputs in timing_inputs.items():
    raw_signal = inputs["raw_signal"]
    raw_background = inputs["raw_background"]
    log_weights = inputs["log_weights"]

    scan_durations, _ = time_repeated(
        lambda: asimov_scan(
            raw_signal,
            raw_background,
            MU_SCAN,
            log_weights=log_weights,
        ),
        FIT_TIMING_REPETITIONS,
        FIT_TIMING_WARMUPS,
    )
    minimum_durations, minimum_outputs = time_repeated(
        lambda: minimize_asimov(
            raw_signal,
            raw_background,
            log_weights=log_weights,
        ),
        FIT_TIMING_REPETITIONS,
        FIT_TIMING_WARMUPS,
    )

    # Vary the seed across end-to-end repetitions so construction is measured
    # on independent quadratures. The proposal and all ratio models stay frozen.
    for warmup in range(END_TO_END_TIMING_WARMUPS):
        construct_quadrature_and_minimize(
            method,
            inputs["sample_size"],
            STUDY_SEED + 30_000 + 1_000 * int(method == "Neural importance") + warmup,
        )

    construction_durations = []
    construction_outputs = []
    for repetition in range(END_TO_END_TIMING_REPETITIONS):
        seed = (
            STUDY_SEED
            + 40_000
            + 1_000 * int(method == "Neural importance")
            + repetition
        )
        synchronize_accelerator()
        start = perf_counter()
        result = construct_quadrature_and_minimize(
            method, inputs["sample_size"], seed
        )
        synchronize_accelerator()
        construction_durations.append(perf_counter() - start)
        construction_outputs.append(result)

    scan_statistics = summarize_durations(scan_durations)
    minimum_statistics = summarize_durations(minimum_durations)
    construction_statistics = summarize_durations(construction_durations)
    precision = inputs["precision"]

    timing_rows.append(
        {
            "method": method,
            "sample_size": int(inputs["sample_size"]),
            "q0_std": float(precision["q0_std"]),
            "q0_variance": float(precision["q0_std"] ** 2),
            "scan_rmse": float(precision["scan_rmse"]),
            "scan_q25_ms": scan_statistics["q25_ms"],
            "scan_median_ms": scan_statistics["median_ms"],
            "scan_q75_ms": scan_statistics["q75_ms"],
            "minimum_q25_ms": minimum_statistics["q25_ms"],
            "minimum_median_ms": minimum_statistics["median_ms"],
            "minimum_q75_ms": minimum_statistics["q75_ms"],
            "minimum_nfev_median": float(
                np.median([result.nfev for result in minimum_outputs])
            ),
            "minimum_mu_hat": float(
                np.median([result.x for result in minimum_outputs])
            ),
            "construction_minimum_q25_ms": construction_statistics["q25_ms"],
            "construction_minimum_median_ms": construction_statistics["median_ms"],
            "construction_minimum_q75_ms": construction_statistics["q75_ms"],
            "construction_minimum_nfev_median": float(
                np.median([result.nfev for result in construction_outputs])
            ),
        }
    )

timing_results = pd.DataFrame(timing_rows)
direct_timing = timing_results[
    timing_results["method"] == "Direct reference"
].iloc[0]
for column in [
    "scan_median_ms",
    "minimum_median_ms",
    "construction_minimum_median_ms",
]:
    timing_results[column.replace("_median_ms", "_speedup")] = (
        float(direct_timing[column]) / timing_results[column]
    )


def format_interval(row, prefix):
    return (
        f"{row[f'{prefix}_median_ms']:.3f} "
        f"[{row[f'{prefix}_q25_ms']:.3f}, "
        f"{row[f'{prefix}_q75_ms']:.3f}]"
    )


formatted_rows = []
for _, row in timing_results.iterrows():
    formatted_rows.append(
        {
            "method": row["method"],
            "Asimov points": f"{int(row['sample_size']):,}",
            "q0 std": f"{row['q0_std']:.6f}",
            "q0 variance": f"{row['q0_variance']:.3e}",
            "scan RMSE": f"{row['scan_rmse']:.6f}",
            "61-point scan ms, median [IQR]": format_interval(row, "scan"),
            "scan speedup": f"{row['scan_speedup']:.2f}x",
            "minimum ms, median [IQR]": format_interval(row, "minimum"),
            "minimum nfev": f"{row['minimum_nfev_median']:.0f}",
            "minimum speedup": f"{row['minimum_speedup']:.2f}x",
            "construction + minimum ms, median [IQR]": format_interval(
                row, "construction_minimum"
            ),
            "end-to-end speedup": (
                f"{row['construction_minimum_speedup']:.2f}x"
            ),
        }
    )

timing_table = pd.DataFrame(formatted_rows)
TIMING_RESULTS_PATH = NIS_CACHE_DIR / "matched_precision_timing.csv"
timing_results.to_csv(TIMING_RESULTS_PATH, index=False)

print(
    "Matched using q0 standard deviation: "
    f"NIS M={MATCHED_NIS_SIZE:,} versus direct M={MATCHED_DIRECT_SIZE:,}."
)
print(
    f"Fit-only timings use {FIT_TIMING_REPETITIONS} repetitions; "
    f"end-to-end timings use {END_TO_END_TIMING_REPETITIONS} repetitions."
)
print(
    "Times are milliseconds; brackets contain the 25th and 75th percentiles."
)
display(timing_table)
print(f"Saved raw timing results to {TIMING_RESULTS_PATH}")


The most portable quantity in this table is the **relative speedup on the same
machine**, rather than the absolute wall time. The scan-only and minimum-only
columns isolate the repeated likelihood workload after a quadrature has been
constructed. The end-to-end column is deliberately more conservative: it also
charges each method for producing a new quadrature and evaluating all frozen
networks needed by that construction.

For an analysis in which one frozen Asimov quadrature is reused for many
profile-likelihood evaluations, the fit-only columns are the relevant
comparison. For a single use followed by disposal of the quadrature, the
end-to-end column is the relevant comparison. Neither comparison charges the
one-time proposal-training cost.


## Comparisons

1. Compare the two variance proxies and independent results across $\epsilon$.
2. Compare globally weighted MLE with variance fine-tuning, using separate checkpoints.
3. Inspect repetitions with large weights or component-normalizer deviations.
4. Compare the v2 figures and saved tables with both previous Exercise 6 runs.
